In [1]:
import pyvisa
from pyvisa.errors import VisaIOError

def check_visa_devices(backend=None, timeout_ms=3000):
    """
    Find available VISA devices and attempt to query their identity.

    Parameters
    ----------
    backend : str or None
        None uses the installed system VISA backend, such as NI-VISA.
        "@py" uses the pure-Python pyvisa-py backend.
    timeout_ms : int
        Communication timeout in milliseconds.

    Returns
    -------
    list of dict
        Detection and communication results for each VISA resource.
    """

    try:
        rm = pyvisa.ResourceManager(backend) if backend else pyvisa.ResourceManager()
    except Exception as exc:
        print("Could not initialise the VISA Resource Manager.")
        print(f"Error: {exc}")
        return []

    print(f"VISA backend: {rm.visalib}")

    try:
        resources = rm.list_resources()
    except Exception as exc:
        print(f"Could not scan for VISA devices: {exc}")
        rm.close()
        return []

    if not resources:
        print("\nNo VISA devices were detected.")
        rm.close()
        return []

    print(f"\nDetected {len(resources)} VISA resource(s):")

    results = []

    for resource_name in resources:
        print(f"\nChecking: {resource_name}")

        result = {
            "resource": resource_name,
            "available": False,
            "identification": None,
            "error": None,
        }

        instrument = None

        try:
            instrument = rm.open_resource(resource_name)
            instrument.timeout = timeout_ms

            result["available"] = True
            print("  Connection: successful")

            try:
                identification = instrument.query("*IDN?").strip()
                result["identification"] = identification
                print(f"  Identification: {identification}")

            except VisaIOError as exc:
                result["error"] = f"Connected, but *IDN? failed: {exc}"
                print("  Device opened, but it did not answer *IDN?.")
                print(f"  Error: {exc}")

            except Exception as exc:
                result["error"] = f"Identification query failed: {exc}"
                print(f"  Identification query failed: {exc}")

        except VisaIOError as exc:
            result["error"] = str(exc)
            print(f"  Connection failed: {exc}")

        except Exception as exc:
            result["error"] = str(exc)
            print(f"  Unexpected error: {exc}")

        finally:
            if instrument is not None:
                instrument.close()

        results.append(result)

    rm.close()
    return results


# First try the VISA implementation installed on the computer,
# such as NI-VISA or Keysight VISA.
results = check_visa_devices()

VISA backend: Visa Library at C:\WINDOWS\system32\visa32.dll

Detected 4 VISA resource(s):

Checking: USB0::0x1AB1::0x0452::MHO9B274902001::INSTR
  Connection: successful
  Identification: RIGOL TECHNOLOGIES,MHO984,MHO9B274902001,00.01.00

Checking: ASRL1::INSTR
  Connection: successful
  Device opened, but it did not answer *IDN?.
  Error: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.

Checking: ASRL3::INSTR
  Connection: successful
  Device opened, but it did not answer *IDN?.
  Error: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.

Checking: ASRL8::INSTR
  Connection: successful
  Device opened, but it did not answer *IDN?.
  Error: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.
